# Spam Email Detection with RNN (LSTM)
This notebook builds an end-to-end spam email classifier using a PyTorch LSTM (a specialized Recurrent Neural Network) and word token embeddings.

In [ ]:
# 1. Import Libraries and Download NLTK resources
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)


In [ ]:
# 2. Load the Dataset and Remove Duplicates
dataset_path = r"C:\Users\pc\OneDrive\Music\Desktop\ML_Projects\Dataset\emails.csv"
df = pd.read_csv(dataset_path)

print("Original Shape:", df.shape)
df.drop_duplicates(inplace=True)
print("Shape after removing duplicates:", df.shape)
df.head()


In [ ]:
# 3. Text Preprocessing (Cleaning, Stopwords Removal, Stemming)
ps = PorterStemmer()
stop_words = set(stopwords.words("english"))

def clean_data(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\r\n", " ", text)
    text = re.sub(r"[^A-Za-z0-9\s]", " ", text)  # Remove Punctuation
    text = re.sub(r"http\S+", " ", text)         # Remove URLS
    text = re.sub(r"[<.*?>]", " ", text)         # Remove HTML tags
    text = text.strip().lower()                  # Convert into lowercase
    
    # Tokenize, remove stopwords, and apply stemming
    tokens = word_tokenize(text)
    cleaned_tokens = [ps.stem(word) for word in tokens if word not in stop_words]
    return " ".join(cleaned_tokens)

print("Cleaning text data...")
df["text"] = df["text"].apply(clean_data)
df.head()


In [ ]:
# 4. Tokenization & Sequence Padding for RNN/LSTM
from collections import Counter

# Build Vocabulary from cleaned text
all_words = []
for text in df["text"]:
    all_words.extend(text.split())

word_counts = Counter(all_words)
vocab = {word: i + 2 for i, (word, _) in enumerate(word_counts.most_common(10000))}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1
vocab_size = len(vocab)

# Convert text to sequence of indices
def text_to_sequence(text, max_len=100):
    seq = [vocab.get(word, 1) for word in text.split()]
    if len(seq) < max_len:
        seq = seq + [0] * (max_len - len(seq))  # Padding
    else:
        seq = seq[:max_len]  # Truncating
    return seq

print("Converting texts to sequences...")
X_seq = np.array([text_to_sequence(t) for t in df["text"]])
y = df["spam"].values

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_seq, y, test_size=0.2, random_state=42)

# Convert to PyTorch Tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.long)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# Create DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)


In [ ]:
# 5. Define PyTorch RNN (LSTM) Model Architecture
class SpamLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(SpamLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        out, (hidden, cell) = self.lstm(embedded)
        # Global max pooling over sequence length
        pooled, _ = torch.max(out, dim=1)
        prediction = self.sigmoid(self.fc(pooled))
        return prediction

embedding_dim = 64
hidden_dim = 64
model = SpamLSTM(vocab_size, embedding_dim, hidden_dim)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# 6. Train the LSTM Model
epochs = 10
model.train()
print("Starting LSTM training...")
for epoch in range(epochs):
    epoch_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss/len(train_loader):.4f}")


In [ ]:
# 7. Evaluate the LSTM Model
model.eval()
with torch.no_grad():
    train_preds = model(X_train_tensor).round()
    test_preds = model(X_test_tensor).round()

train_acc = accuracy_score(y_train, train_preds.numpy())
test_acc = accuracy_score(y_test, test_preds.numpy())

print(f"Train Accuracy: {train_acc * 100:.2f}%")
print(f"Test Accuracy: {test_acc * 100:.2f}%")


In [ ]:
# 8. Predict on Custom Email Messages using LSTM
def predict_spam(message):
    cleaned = clean_data(message)
    seq = text_to_sequence(cleaned)
    tensor_input = torch.tensor([seq], dtype=torch.long)
    model.eval()
    with torch.no_grad():
        pred_prob = model(tensor_input).item()
    classification = "Spam" if pred_prob >= 0.5 else "Ham"
    print(f"Message: {message}")
    print(f"Prediction: {classification} (Spam Probability: {pred_prob:.4f})\n")

# Test custom inputs
predict_spam("Congratulations! You won a free vacation to the Bahamas. Click here to claim your prize.")
predict_spam("Hey, are we still meeting for lunch today at 12:30?")
